# Advanced Problems: `iter()`, Iterables, Iterators, and Sequence Protocols

This notebook contains advanced practice problems **with complete solutions** for Python's iteration model.

Topics covered:

- `iter(obj)` and `next(iterator)`
- iterable protocol: `__iter__`
- iterator protocol: `__iter__` + `__next__`
- sequence fallback protocol: `__getitem__`
- `StopIteration` and `IndexError`
- EAFP style: "ask forgiveness, not permission"
- sentinel iterators: `iter(callable, sentinel)`
- reusable iterables vs one-shot iterators
- lazy custom iterators
- robust testing with `assert`

## Setup: helper functions used in several examples

In [1]:
from collections.abc import Iterable, Iterator
from itertools import islice
from typing import Any


def show(title, value):
    """Small display helper for examples."""
    print(f"{title}: {value}")


def take(n, iterable):
    """Return at most n items from any iterable."""
    return list(islice(iterable, n))


def is_iterable(obj):
    """Best practical iterable test: try to obtain an iterator."""
    try:
        iter(obj)
        return True
    except TypeError:
        return False


def is_iterator(obj):
    """
    An iterator must:
    1. be iterable
    2. return itself from iter(obj)
    3. implement __next__
    """
    try:
        return iter(obj) is obj and hasattr(obj, "__next__")
    except TypeError:
        return False

## Problem 1 — Diagnose Iterable vs Iterator Behavior

You are given several objects. For each one, determine:

1. Is it iterable?
2. Is it an iterator?
3. Can it be iterated more than once from the beginning?

Write code that demonstrates the differences.

Objects to test:

- list
- tuple
- string
- dictionary
- list iterator
- generator expression
- custom sequence using `__getitem__`

In [2]:
# Solution 1

class OnlyGetItemSquares:
    """Sequence-style object: iterable via __getitem__ fallback."""
    def __init__(self, n):
        self.n = n

    def __getitem__(self, index):
        if index < 0:
            raise IndexError("negative indexes are not supported here")
        if index >= self.n:
            raise IndexError
        return index ** 2


objects = {
    "list": [10, 20, 30],
    "tuple": (10, 20, 30),
    "string": "abc",
    "dict": {"a": 1, "b": 2},
    "list_iterator": iter([10, 20, 30]),
    "generator_expression": (x * 10 for x in range(3)),
    "custom_getitem_sequence": OnlyGetItemSquares(4),
}

for name, obj in objects.items():
    print(f"\n{name}")
    print("-" * len(name))
    print("type:", type(obj).__name__)
    print("is_iterable:", is_iterable(obj))
    print("is_iterator:", is_iterator(obj))

    first_pass = list(obj)
    second_pass = list(obj)

    print("first pass :", first_pass)
    print("second pass:", second_pass)

# Key observation:
# Reusable iterables such as lists, tuples, strings, dicts, and custom sequences
# create a fresh iterator every time.
#
# One-shot iterators such as list_iterator and generator_expression are consumed.


list
----
type: list
is_iterable: True
is_iterator: False
first pass : [10, 20, 30]
second pass: [10, 20, 30]

tuple
-----
type: tuple
is_iterable: True
is_iterator: False
first pass : [10, 20, 30]
second pass: [10, 20, 30]

string
------
type: str
is_iterable: True
is_iterator: False
first pass : ['a', 'b', 'c']
second pass: ['a', 'b', 'c']

dict
----
type: dict
is_iterable: True
is_iterator: False
first pass : ['a', 'b']
second pass: ['a', 'b']

list_iterator
-------------
type: list_iterator
is_iterable: True
is_iterator: True
first pass : [10, 20, 30]
second pass: []

generator_expression
--------------------
type: generator
is_iterable: True
is_iterator: True
first pass : [0, 10, 20]
second pass: []

custom_getitem_sequence
-----------------------
type: OnlyGetItemSquares
is_iterable: True
is_iterator: False
first pass : [0, 1, 4, 9]
second pass: [0, 1, 4, 9]


## Problem 2 — Build a Correct Reusable Iterable

Implement `PowersOfTwo(n)` so that:

- `list(PowersOfTwo(5)) == [1, 2, 4, 8, 16]`
- multiple loops over the same object restart from the beginning
- `iter(obj)` returns a **new independent iterator** each time
- the class itself is **not** an iterator

In [3]:
# Solution 2

class PowersOfTwo:
    """
    Reusable iterable.
    Each call to iter(obj) returns a new generator iterator.
    """
    def __init__(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        self.n = n

    def __iter__(self):
        for exponent in range(self.n):
            yield 2 ** exponent


p = PowersOfTwo(5)

assert list(p) == [1, 2, 4, 8, 16]
assert list(p) == [1, 2, 4, 8, 16]

it1 = iter(p)
it2 = iter(p)

assert next(it1) == 1
assert next(it1) == 2
assert next(it2) == 1      # independent iterator
assert iter(p) is not p    # iterable is not its own iterator

print("PowersOfTwo works correctly.")
print(list(p))

PowersOfTwo works correctly.
[1, 2, 4, 8, 16]


## Problem 3 — Build a Manual Iterator Class

Implement the same behavior as Problem 2, but this time use two classes:

- `PowersOfTwoIterable`
- `PowersOfTwoIterator`

Requirements:

- `PowersOfTwoIterable.__iter__` returns a fresh `PowersOfTwoIterator`
- `PowersOfTwoIterator.__iter__` returns `self`
- `PowersOfTwoIterator.__next__` raises `StopIteration` correctly

In [4]:
# Solution 3

class PowersOfTwoIterator:
    def __init__(self, n):
        self.n = n
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.n:
            raise StopIteration

        value = 2 ** self.current
        self.current += 1
        return value


class PowersOfTwoIterable:
    def __init__(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        self.n = n

    def __iter__(self):
        return PowersOfTwoIterator(self.n)


powers = PowersOfTwoIterable(6)

assert list(powers) == [1, 2, 4, 8, 16, 32]
assert list(powers) == [1, 2, 4, 8, 16, 32]

a = iter(powers)
b = iter(powers)

assert is_iterator(a)
assert next(a) == 1
assert next(a) == 2
assert next(b) == 1

print("Manual iterable + iterator pair works.")

Manual iterable + iterator pair works.


## Problem 4 — Sequence Fallback with `__getitem__`

Create a `SquaresSequence` class that supports iteration **without defining `__iter__`**.

Requirements:

- `SquaresSequence(5)[0] == 0`
- `SquaresSequence(5)[4] == 16`
- iteration works because Python falls back to `__getitem__`
- out-of-range integer indexes raise `IndexError`
- negative indexes are supported
- slices are supported

In [5]:
# Solution 4

class SquaresSequence:
    def __init__(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, index):
        if isinstance(index, slice):
            start, stop, step = index.indices(self.n)
            return [i ** 2 for i in range(start, stop, step)]

        if not isinstance(index, int):
            raise TypeError("index must be an int or slice")

        if index < 0:
            index += self.n

        if index < 0 or index >= self.n:
            raise IndexError("index out of range")

        return index ** 2


sq = SquaresSequence(5)

assert sq[0] == 0
assert sq[4] == 16
assert sq[-1] == 16
assert sq[-2] == 9
assert sq[1:4] == [1, 4, 9]
assert list(sq) == [0, 1, 4, 9, 16]

print("SquaresSequence is iterable via __getitem__ fallback.")
print(list(sq))

SquaresSequence is iterable via __getitem__ fallback.
[0, 1, 4, 9, 16]


## Problem 5 — Fix a Broken Iterable

The class below appears to define `__iter__`, but `iter(obj)` fails.

Your tasks:

1. Explain the bug.
2. Fix the class.
3. Add tests proving the fixed object is iterable.

In [6]:
# Broken version

class BrokenCountdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return "not an iterator"


broken = BrokenCountdown(3)

try:
    iter(broken)
except TypeError as ex:
    print("Expected failure:", ex)

Expected failure: iter() returned non-iterator of type 'str'


In [7]:
# Solution 5

class CountdownIterator:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration

        value = self.current
        self.current -= 1
        return value


class Countdown:
    def __init__(self, start):
        if start < 0:
            raise ValueError("start must be non-negative")
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)


countdown = Countdown(3)

assert list(countdown) == [3, 2, 1, 0]
assert list(countdown) == [3, 2, 1, 0]
assert is_iterable(countdown)
assert not is_iterator(countdown)

it = iter(countdown)
assert is_iterator(it)
assert list(it) == [3, 2, 1, 0]
assert list(it) == []      # same iterator is exhausted

print("BrokenCountdown fixed as Countdown.")

BrokenCountdown fixed as Countdown.


## Problem 6 — Use `iter(callable, sentinel)`

Python has a two-argument form:

```python
iter(callable, sentinel)
```

It repeatedly calls `callable()` until the returned value equals `sentinel`.

Build examples that use this form to:

1. Read chunks from a fake file reader until an empty string is returned.
2. Roll a deterministic die until 6 appears.

In [8]:
# Solution 6.1: fake file reader

class FakeFile:
    def __init__(self, chunks):
        self.chunks = list(chunks)
        self.position = 0

    def read(self, size):
        if self.position >= len(self.chunks):
            return ""

        chunk = self.chunks[self.position]
        self.position += 1
        return chunk


fake_file = FakeFile(["alpha", "beta", "gamma"])

# The lambda creates a zero-argument callable,
# because iter(callable, sentinel) calls callable with no arguments.
chunks = list(iter(lambda: fake_file.read(5), ""))

assert chunks == ["alpha", "beta", "gamma"]
print(chunks)

['alpha', 'beta', 'gamma']


In [9]:
# Solution 6.2: deterministic die roller

class DeterministicDie:
    def __init__(self, rolls):
        self.rolls = iter(rolls)

    def roll(self):
        return next(self.rolls)


die = DeterministicDie([2, 4, 1, 5, 6, 3, 2])

rolls_before_six = list(iter(die.roll, 6))

assert rolls_before_six == [2, 4, 1, 5]
print("rolls before first 6:", rolls_before_six)

rolls before first 6: [2, 4, 1, 5]


## Problem 7 — Create a Lazy `take_until` Iterator

Write a function `take_until(iterable, stop_value)` that lazily yields items from `iterable` until `stop_value` is found.

Requirements:

- The stop value is **not** yielded.
- The function should work with infinite iterables.
- The function should not convert the input to a list.

In [10]:
# Solution 7

def take_until(iterable, stop_value):
    for item in iterable:
        if item == stop_value:
            return
        yield item


def natural_numbers():
    n = 0
    while True:
        yield n
        n += 1


assert list(take_until([1, 2, 3, 99, 4], 99)) == [1, 2, 3]
assert take(5, take_until(natural_numbers(), 10)) == [0, 1, 2, 3, 4]
assert list(take_until(natural_numbers(), 5)) == [0, 1, 2, 3, 4]

print("take_until works lazily.")

take_until works lazily.


## Problem 8 — Implement a Peekable Iterator

Create a `Peekable` class that wraps any iterable and supports:

- `peek()` to look at the next item without consuming it
- `next(obj)` to consume the next item
- iteration in a `for` loop
- correct `StopIteration` behavior

This is useful when parsing streams.

In [11]:
# Solution 8

_SENTINEL = object()


class Peekable:
    def __init__(self, iterable):
        self._iterator = iter(iterable)
        self._cache = _SENTINEL

    def __iter__(self):
        return self

    def __next__(self):
        if self._cache is not _SENTINEL:
            value = self._cache
            self._cache = _SENTINEL
            return value

        return next(self._iterator)

    def peek(self):
        if self._cache is _SENTINEL:
            self._cache = next(self._iterator)
        return self._cache


p = Peekable([10, 20, 30])

assert p.peek() == 10
assert p.peek() == 10
assert next(p) == 10
assert p.peek() == 20
assert list(p) == [20, 30]

try:
    p.peek()
except StopIteration:
    print("Correctly raises StopIteration when peeking exhausted iterator.")

print("Peekable works.")

Correctly raises StopIteration when peeking exhausted iterator.
Peekable works.


## Problem 9 — Implement Lazy Flattening

Write `flatten_once(iterable)` that flattens one level of nesting.

Requirements:

- `flatten_once([[1, 2], [3, 4]])` yields `1, 2, 3, 4`
- strings and bytes should be treated as scalar values, not expanded
- non-iterable items should be yielded as-is
- implementation should use EAFP style

In [12]:
# Solution 9

def flatten_once(iterable):
    for item in iterable:
        if isinstance(item, (str, bytes)):
            yield item
            continue

        try:
            iterator = iter(item)
        except TypeError:
            yield item
        else:
            yield from iterator


data = [[1, 2], (3, 4), "abc", b"xy", 5, {"k1": 10, "k2": 20}]
result = list(flatten_once(data))

# Dictionaries iterate over keys by default.
assert result == [1, 2, 3, 4, "abc", b"xy", 5, "k1", "k2"]

print(result)

[1, 2, 3, 4, 'abc', b'xy', 5, 'k1', 'k2']


## Problem 10 — Implement Round-Robin Iteration

Create `round_robin(*iterables)` that yields one item from each iterable in turn.

Example:

```python
list(round_robin("ABC", [1, 2], ("x", "y", "z")))
# ['A', 1, 'x', 'B', 2, 'y', 'C', 'z']
```

Requirements:

- Works with iterables of different lengths.
- Exhausted iterators are removed.
- Does not pre-convert inputs to lists.

In [13]:
# Solution 10

from collections import deque


def round_robin(*iterables):
    iterators = deque(iter(it) for it in iterables)

    while iterators:
        iterator = iterators.popleft()

        try:
            value = next(iterator)
        except StopIteration:
            continue

        yield value
        iterators.append(iterator)


assert list(round_robin("ABC", [1, 2], ("x", "y", "z"))) == [
    "A", 1, "x", "B", 2, "y", "C", "z"
]

assert list(round_robin([], [1, 2, 3], "ab")) == [1, "a", 2, "b", 3]

print(list(round_robin("ABC", [1, 2], ("x", "y", "z"))))

['A', 1, 'x', 'B', 2, 'y', 'C', 'z']


## Problem 11 — Detect Subtle Consumption Bugs

The function below has a bug:

```python
def average_first_three(iterable):
    if not is_iterable(iterable):
        raise TypeError("expected iterable")
    iterator = iter(iterable)
    return sum(iterator) / 3
```

Tasks:

1. Explain why this is wrong.
2. Write a correct version that averages only the first three items.
3. Raise `ValueError` if fewer than three items are available.

In [14]:
# Solution 11

def average_first_three(iterable):
    iterator = iter(iterable)
    values = []

    for _ in range(3):
        try:
            values.append(next(iterator))
        except StopIteration:
            raise ValueError("expected at least three values") from None

    return sum(values) / 3


assert average_first_three([10, 20, 30, 1000]) == 20
assert average_first_three(iter([3, 6, 9, 12])) == 6

try:
    average_first_three([1, 2])
except ValueError as ex:
    print("Expected:", ex)

# Why the original was wrong:
# - It sums the entire iterator, not just the first three values.
# - It does not verify that exactly three values were available.
# - Calling is_iterable is unnecessary because iter(iterable) already validates.

Expected: expected at least three values


## Problem 12 — Build a Paginated API Iterator

Simulate an API that returns records page by page.

Your task:

- Create `PaginatedRecords` as a reusable iterable.
- Each iterator should lazily fetch pages.
- It should stop when the API returns an empty page.
- Multiple loops over `PaginatedRecords` should fetch from the start again.

In [15]:
# Solution 12

class FakeAPI:
    def __init__(self, pages):
        self.pages = list(pages)
        self.calls = []

    def fetch_page(self, page_number):
        self.calls.append(page_number)

        if page_number >= len(self.pages):
            return []

        return self.pages[page_number]


class PaginatedRecordsIterator:
    def __init__(self, api):
        self.api = api
        self.page_number = 0
        self.buffer = []
        self.finished = False

    def __iter__(self):
        return self

    def __next__(self):
        while not self.buffer:
            if self.finished:
                raise StopIteration

            page = self.api.fetch_page(self.page_number)
            self.page_number += 1

            if not page:
                self.finished = True
                raise StopIteration

            self.buffer.extend(page)

        return self.buffer.pop(0)


class PaginatedRecords:
    def __init__(self, api):
        self.api = api

    def __iter__(self):
        return PaginatedRecordsIterator(self.api)


api = FakeAPI([
    [{"id": 1}, {"id": 2}],
    [{"id": 3}],
    [{"id": 4}, {"id": 5}],
    [],
])

records = PaginatedRecords(api)

assert list(records) == [{"id": 1}, {"id": 2}, {"id": 3}, {"id": 4}, {"id": 5}]
assert list(records) == [{"id": 1}, {"id": 2}, {"id": 3}, {"id": 4}, {"id": 5}]

# Each full pass fetches pages 0, 1, 2, and then page 3,
# where the empty page signals the end.
assert api.calls == [0, 1, 2, 3, 0, 1, 2, 3]

print("API calls:", api.calls)
print("records:", list(records))

API calls: [0, 1, 2, 3, 0, 1, 2, 3]
records: [{'id': 1}, {'id': 2}, {'id': 3}, {'id': 4}, {'id': 5}]


## Problem 13 — Implement `enumerate` Manually

Create `my_enumerate(iterable, start=0)`.

Requirements:

- Must be lazy.
- Must work with one-shot iterators.
- Must not use the built-in `enumerate`.

In [16]:
# Solution 13

def my_enumerate(iterable, start=0):
    index = start
    for item in iterable:
        yield index, item
        index += 1


assert list(my_enumerate(["a", "b", "c"])) == [(0, "a"), (1, "b"), (2, "c")]
assert list(my_enumerate(["a", "b"], start=10)) == [(10, "a"), (11, "b")]

one_shot = (x * x for x in range(3))
assert list(my_enumerate(one_shot, start=1)) == [(1, 0), (2, 1), (3, 4)]
assert list(one_shot) == []

print(list(my_enumerate("xyz", start=5)))

[(5, 'x'), (6, 'y'), (7, 'z')]


## Problem 14 — Implement a Window Iterator

Create `sliding_window(iterable, size)`.

Example:

```python
list(sliding_window([1, 2, 3, 4], 3))
# [(1, 2, 3), (2, 3, 4)]
```

Requirements:

- Must be lazy.
- Must work with any iterable.
- If `size <= 0`, raise `ValueError`.
- If there are fewer than `size` elements, yield nothing.

In [17]:
# Solution 14

from collections import deque


def sliding_window(iterable, size):
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)
    window = deque(maxlen=size)

    for _ in range(size):
        try:
            window.append(next(iterator))
        except StopIteration:
            return

    yield tuple(window)

    for item in iterator:
        window.append(item)
        yield tuple(window)


assert list(sliding_window([1, 2, 3, 4], 3)) == [(1, 2, 3), (2, 3, 4)]
assert list(sliding_window([1, 2], 3)) == []
assert list(sliding_window("abcd", 2)) == [("a", "b"), ("b", "c"), ("c", "d")]

try:
    list(sliding_window([1, 2, 3], 0))
except ValueError as ex:
    print("Expected:", ex)

print(list(sliding_window(range(6), 4)))

Expected: size must be positive
[(0, 1, 2, 3), (1, 2, 3, 4), (2, 3, 4, 5)]


## Problem 15 — Build a Resettable Iterator Carefully

A resettable iterator is often a code smell, because reusable iterables are usually better.

Still, implement `ResettableIterator` that:

- wraps a finite iterable
- supports `next(obj)`
- supports `reset()`
- uses an internal cached tuple so it can restart

In [18]:
# Solution 15

class ResettableIterator:
    def __init__(self, iterable):
        self._items = tuple(iterable)
        self._index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._index >= len(self._items):
            raise StopIteration

        value = self._items[self._index]
        self._index += 1
        return value

    def reset(self):
        self._index = 0
        return self


r = ResettableIterator(x * 10 for x in range(4))

assert list(r) == [0, 10, 20, 30]
assert list(r) == []

r.reset()
assert next(r) == 0
assert next(r) == 10

r.reset()
assert list(r) == [0, 10, 20, 30]

print("ResettableIterator works, but stores all items in memory.")

ResettableIterator works, but stores all items in memory.


## Problem 16 — Iterator Best-Practice Checklist

Use this checklist when designing iterator-heavy code:

1. If users should be able to loop multiple times, create a **reusable iterable**.
2. If state is consumed over time, create an **iterator**.
3. `__iter__` on an iterator should return `self`.
4. `__next__` should raise `StopIteration`, not return a magic value.
5. A sequence can be iterable with only `__getitem__`, but an explicit `__iter__` is often clearer.
6. Prefer EAFP: attempt `iter(obj)` or attempt the loop and catch `TypeError` only when you have a meaningful recovery.
7. Do not pre-convert to `list` unless you intentionally need caching, indexing, repeated traversal, or length.
8. Avoid sharing one iterator across unrelated consumers unless shared consumption is intended.